# Brightway 2.5 cheatsheet in practice — comparing photovoltaic panels

This notebook walks through **every command block of the BW 2.5 beginner cheatsheet**, executed against a
real case study instead of toy snippets. Each section is headed with the cheatsheet slide it corresponds to.

**Case study.** Using the **BAFU** database (Swiss Federal Office for the Environment) we compare
photovoltaic panels along two axes:

1. **Panel technology** — a-Si, CIS, micro-Si, multi-Si, perovskite-Si tandem, ribbon-Si, single-Si
2. **Manufacturing geography** — single-Si and multi-Si panels made in RER, CN, APAC and US

Functional unit throughout: **1 m² of photovoltaic panel, at plant**.

> ⚠️ **Corrections to the cheatsheet.** Several cheatsheet snippets do not run on
> `bw2data 4.7` / `bw2calc 2.5.0`. Where that happens this notebook shows the failing form,
> explains why, and gives the working replacement. Section 15 collects all of them in one table.

## 0) Everything around BW — before you open a notebook
*(cheatsheet slides: "Everything around BW", "Installing, opening, upgrading bw")*

These run in the **conda prompt**, not here. The cheatsheet versions are reproduced with two fixes:

```bash
# Create the environment. NOTE: the cheatsheet renders "-c" as an en-dash "–c".
# It must be an ASCII hyphen, otherwise conda errors out. Copying from PowerPoint
# often silently converts hyphens to en-dashes, which is the single most common
# "the cheatsheet command doesn't work" report.
conda create -n yourenv -c conda-forge -c cmutel brightway25

# Better for this course: build the environment from the pinned course file,
# which also brings jupyterlab, matplotlib, seaborn, polyviz, wurst and edges.
conda env create -f envs/bw_env_win64.yaml

# Every working session starts here
conda activate bw
jupyter lab

# Updating
conda update -c conda-forge -c cmutel brightway25

# Activity Browser (its own environment)
conda create -n ab -c conda-forge activity-browser
conda activate ab
activity-browser
```

**Freezing an environment** — the cheatsheet's line is missing the `>` redirect:

```bash
conda list -n bw --export > C:\yourpath\bw_20260916.txt   # exact build pins, same platform only
conda env export -n bw > C:\yourpath\bw_20260916.yml      # portable; recreate with the line below
conda env create -f C:\yourpath\bw_20260916.yml
```

## 1) Starting to use bw — imports
*(cheatsheet slide: "Starting to use bw – in jupyter notebook")*

In [ ]:
# the three brightway packages the cheatsheet lists as the minimum
import bw2data as bd
import bw2calc as bc
import bw2io as bi

# usually needed on top
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from pathlib import Path
from pprint import pprint
import inspect
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("bw2data ", bd.__version__)
print("bw2calc ", bc.__version__)
print("bw2io   ", bi.__version__)

### Plot styling used throughout

One place to define the look, so every figure below is consistent. The palette is a validated
categorical set (slots assigned in fixed order, never cycled) plus a single-hue blue ramp for
magnitude.

In [ ]:
# --- chart theme -------------------------------------------------------------
SURFACE = "#fcfcfb"
INK     = "#0b0b0b"
INK_2   = "#52514e"
GRID    = "#e3e2de"

# categorical slots, assigned in fixed order (identity encoding)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

# single-hue sequential ramp (magnitude encoding)
BLUES = LinearSegmentedColormap.from_list(
    "bw_blues",
    ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"],
)

plt.rcParams.update({
    "figure.facecolor":   SURFACE,
    "axes.facecolor":     SURFACE,
    "savefig.facecolor":  SURFACE,
    "axes.edgecolor":     GRID,
    "axes.labelcolor":    INK_2,
    "axes.titlecolor":    INK,
    "axes.titlesize":     12,
    "axes.titleweight":   "semibold",
    "axes.titlelocation": "left",
    "axes.titlepad":      12,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "xtick.color":        INK_2,
    "ytick.color":        INK_2,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "grid.color":         GRID,
    "grid.linewidth":     0.8,
    "legend.frameon":     False,
    "legend.fontsize":    9,
    "font.size":          10,
    "figure.dpi":         110,
})

def tidy(ax, xgrid=False, ygrid=True):
    """Hairline, recessive grid on one axis only."""
    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=ygrid)
    ax.grid(axis="x", visible=xgrid)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_linewidth(0.8)
    return ax

print("theme ready")

## 2) Working on projects
*(cheatsheet slide: "Working on projects – in jupyter notebook")*

The cheatsheet's tip still holds: type `bd.projects.` then **Tab** to list everything, and append
`?` to any function to read its docstring.

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")

print("current project :", bd.projects.current)
print("project dir     :", bd.projects.dir)
print("n projects      :", len(list(bd.projects)))

In [ ]:
# which projects do I have?
[p.name for p in bd.projects]

In [ ]:
# general information about all projects
bd.projects.report()

### Create, copy, rename, delete a project

```python
bd.projects.set_current("myproject")          # activates, and creates it first if missing
bd.projects.create_project("myproject")       # creates only; you stay where you are

bd.projects.copy_project("newname", switch=True)
bd.projects.rename_project("newname")         # renames the *current* project, in place
bd.projects.delete_project("myproject", delete_dir=True)
```

> ❌ **Cheatsheet error.** The cheatsheet writes `bd.projects.copy_projects(...)` (plural).
> The method is `copy_project` — singular. The plural name does not exist.
>
> ❌ **Cheatsheet error.** The cheatsheet says *"You need to copy your project if you want to
> rename it, and delete the old one."* That was true in bw2; since `bw2data 4.x` there is a
> direct `bd.projects.rename_project(new_name)`.

The cell below proves both, without touching your course project.

In [ ]:
assert not hasattr(bd.projects, "copy_projects"), "unexpected: plural form exists"
print("bd.projects.copy_projects  -> does NOT exist (cheatsheet typo)")
print("bd.projects.copy_project   -> exists:", hasattr(bd.projects, "copy_project"))
print("bd.projects.rename_project -> exists:", hasattr(bd.projects, "rename_project"))

for name in ("copy_project", "rename_project", "delete_project", "create_project"):
    print(f"  {name}{inspect.signature(getattr(bd.projects, name))}")

In [ ]:
# a throwaway project, so the demonstration is safe
bd.projects.set_current("cheatsheet-scratch")
print("now in:", bd.projects.current)

bd.projects.copy_project("cheatsheet-scratch-copy", switch=True)
print("copied to and switched to:", bd.projects.current)

bd.projects.delete_project("cheatsheet-scratch-copy", delete_dir=True)
print("deleted the copy; now in:", bd.projects.current)

## 3) Manage databases
*(cheatsheet slide: "Manage databases")*

All four cheatsheet operations, run on a one-activity dummy database inside the scratch project.

In [ ]:
bd.projects.set_current("cheatsheet-scratch")

# a minimal database so we have something to copy / rename / delete
bd.Database("mydb").write({
    ("mydb", "a"): {
        "name": "Dummy activity",
        "unit": "kilogram",
        "location": "CH",
        "exchanges": [{"input": ("mydb", "a"), "amount": 1, "type": "production"}],
    }
})

print("which dbs do I have? ", list(bd.databases))

original = bd.Database("mydb")
copy = original.copy("mydb_newname")           # copy
print("after copy           :", list(bd.databases))

copy.rename("mydb_renamed")                    # rename
print("after rename         :", list(bd.databases))

del bd.databases["mydb_renamed"]               # delete
print("after delete         :", list(bd.databases))

print("len / type           :", len(original), type(original).__name__)

In [ ]:
# back to the real project for the rest of the notebook
bd.projects.set_current("aalborg-rlcia-2026")
bd.projects.delete_project("cheatsheet-scratch", delete_dir=True)
bd.projects.set_current("aalborg-rlcia-2026")

print("current project:", bd.projects.current)
print("databases      :", list(bd.databases))
print("LCIA methods   :", len(bd.methods))

## 4) Import databases
*(cheatsheet slides: "Import databases – biosphere, methods, ecoinvent" and "– own data / premise")*

The commands below are shown but **guarded**, because this project already contains everything we need.
Un-guard them only in a fresh project.

```python
# a. biosphere + ecoinvent in one call (needs ecoinvent credentials)
bi.import_ecoinvent_release("3.11", "cutoff", "ecoinvent_user", "ecoinvent_password")
#    system model: cutoff / apos / consequential / EN15804

# b1. biosphere + LCIA methods only
bi.bw2setup()

# b2. any ecoSpold2 folder
fp = r"C:\ecoinvent_3.11_cutoff_ecoSpold02\datasets"
imp = bi.SingleOutputEcospold2Importer(fp, "ev311cutoff")
imp.apply_strategies()
imp.statistics()
if len(list(imp.unlinked)) == 0:      # <- see the fix note below
    imp.write_database()
```

> ❌ **Cheatsheet error (bracket in the wrong place).** The cheatsheet writes
> `if len(list(ei311imp.unlinked) == 0):`. The closing bracket sits after `unlinked` instead of
> after the `list(...)` call, so Python evaluates `list(...) == 0` first — a `bool` — and then
> calls `len()` on it, raising `TypeError: object of type 'bool' has no len()`. It fails this way
> whether or not there are unlinked exchanges. The bracket belongs after `unlinked`:
> `if len(list(imp.unlinked)) == 0:`.

In [ ]:
# BAFU comes as an Excel workbook shipped with this repository.
# It is already imported in this project, so we only re-import if it is missing.
bafu_file = Path("../../data/lci-bafu.xlsx")

if "bafu" not in bd.databases:
    bi.create_core_migrations()

    importer = bi.ExcelImporter(str(bafu_file))
    importer.apply_strategies()
    importer.match_database(fields=("name", "reference product", "location"))
    importer.match_database("ecoinvent-3.10-biosphere", fields=("name", "unit", "categories"))
    importer.statistics()

    # inspect before writing; only write when nothing is unlinked
    if len(list(importer.unlinked)) == 0:
        importer.write_database()
    else:
        # importer.write_excel()                      # whole inventory, unlinked rows in red
        # importer.write_excel(only_unlinked=True)    # only the problem rows
        # importer.drop_unlinked(i_am_reckless=True)  # ONLY if you know they are dispensable
        raise RuntimeError(f"{len(list(importer.unlinked))} unlinked exchanges — fix before writing")
else:
    print("bafu already imported — skipping")

bafu = bd.Database("bafu")
bio = bd.Database("ecoinvent-3.10-biosphere")

print("activities in bafu :", len(bafu))
print("flows in biosphere :", len(bio))

## 5) Checking the content of the databases
*(cheatsheet slide: "Checking the content of the dbs")*

In [ ]:
# the three objects the cheatsheet introduces
db = bd.Database("bafu")
bio = bd.Database("ecoinvent-3.10-biosphere")
m = bd.methods

print("len(db)   :", len(db),  "| type:", type(db).__name__)
print("len(bio)  :", len(bio), "| type:", type(bio).__name__)
print("len(m)    :", len(m),   "| type:", type(m).__name__)

In [ ]:
# picking a random activity, and looking at it
act = db.random()
act.as_dict()

In [ ]:
# which fields can I search on? -> the keys of any activity
list(act.keys())

In [ ]:
# which compartments exist in the biosphere?
sorted(set(f["categories"] for f in bio))

### Searching — the two approaches from the cheatsheet

**a. the `search` function** (fast, fuzzy, full-text)

In [ ]:
pd.DataFrame(
    [{"name": a["name"], "location": a["location"], "unit": a["unit"]}
     for a in db.search("photovoltaic panel", limit=8)]
)

In [ ]:
bio.search("carbon dioxide", filter={"categories": "urban", "name": "fossil"})

**b. list comprehensions** (explicit, exact, combinable — this is what we use to build the case study)

In [ ]:
# biosphere example from the cheatsheet
co2 = [
    flow for flow in bio
    if "Carbon dioxide" in flow["name"]
    and ", fossil" not in flow["name"]
    and flow["categories"] == ("soil",)
]
co2

In [ ]:
# technosphere example: coal electricity in Germany
coal_de = [
    a for a in db
    if "electricity" in a["name"].lower()
    and "coal" in a["name"].lower()
    and a["location"] == "DE"
]
pd.DataFrame([{"name": a["name"], "location": a["location"], "unit": a["unit"]} for a in coal_de]).head(10)

In [ ]:
# methods example
ef31 = [mt for mt in bd.methods if "EF v3.1" in str(mt) and "no LT" not in str(mt)]
print(len(ef31), "EF v3.1 methods (long-term excluded)")
ef31[:5]

> ❌ **Cheatsheet error.** The cheatsheet then writes `bd.Method(ilcd).metadata['unit']` — here
> `bd.Method(ef31).metadata` — but the comprehension returns a **list** of method tuples, and
> `bd.Method` needs one hashable tuple. You have to index into the list first. The cheatsheet's
> own prose says so ("you can choose a list element with e.g. `[1]`"), but the code line does not.
>
> *(The cheatsheet's example searches `'ILCD 2.0'`; we use **EF v3.1** throughout, since that is
> the method family shipped with the course project.)*

In [ ]:
try:
    bd.Method(ef31).metadata                      # as printed on the cheatsheet
except TypeError as err:
    print("FAILS as written ->", type(err).__name__, err)

# correct: index into the list first
print("\nworks:")
pprint(bd.Method(ef31[0]).metadata)
print("\nunit:", bd.Method(ef31[0]).metadata["unit"])

### Diving deeper — the exchanges of an activity

In [ ]:
panel = [a for a in db
         if a["name"] == "Photovoltaic panel, single-Si, at plant"
         and a["location"] == "RER"][0]
panel

In [ ]:
# a. the three exchange families as lists
print("production   :", len(list(panel.production())))
print("technosphere :", len(list(panel.technosphere())))
print("biosphere    :", len(list(panel.biosphere())))

In [ ]:
# b. the explicit loop from the cheatsheet
for e in panel.exchanges():
    if e["type"] == "technosphere":
        print(f"{e['amount']:>10.5f}  {e.input['unit']:<14}  {e['name'][:52]:<52}  {e.input['location']}")

> ℹ️ **Cheatsheet note.** The cheatsheet uses `e.get('location')` inside this loop. It happens to
> work, because an `Exchange` proxy falls through to its *input* node for keys it does not carry
> itself — but that is easy to misread. Writing `e.input["location"]` and `e.input["unit"]` makes
> it explicit that you are asking about the **input node**, not about the exchange.

### A more convenient view: whole databases as DataFrames

Not on the cheatsheet, but worth knowing — `bw2data 4.x` hands you nodes and edges directly
as pandas DataFrames, which makes exploratory filtering much faster than looping.

In [ ]:
nodes = db.nodes_to_dataframe()
print(nodes.shape)
nodes.head(3)

In [ ]:
# every photovoltaic panel dataset in BAFU, in one line
pv = nodes[nodes["name"].str.contains("Photovoltaic panel", case=False, na=False)]
pv[["name", "location", "unit"]].sort_values(["name", "location"]).head(25)

---
## 6) Building the case study

Two comparison sets, both with the functional unit **1 m² of panel, at plant**.

**Set A — panel technology.** BAFU carries each technology at its dominant real production
location, so the geographies differ between rows. That is *realistic* but it means a row
difference mixes technology and manufacturing electricity. We separate the two effects in
sections 10 and 11.

**Set B — manufacturing geography.** The two technologies BAFU provides in four regions each
(RER, CN, APAC, US), so technology is held constant and only the location varies.

In [ ]:
def one(name, location, database="bafu"):
    """Fetch exactly one activity, and fail loudly if the query is ambiguous."""
    hits = [a for a in bd.Database(database) if a["name"] == name and a["location"] == location]
    if len(hits) != 1:
        raise LookupError(f"{len(hits)} hits for {name!r} @ {location!r}")
    return hits[0]


# Set A - panel technology (location = where BAFU models that technology)
TECHNOLOGY = {
    "a-Si":          ("Photovoltaic panel, a-Si, at plant",                 "US"),
    "CIS":           ("Photovoltaic panel, CIS, at plant",                  "DE"),
    "micro-Si":      ("Photovoltaic panel, micro-Si, at plant",             "CN"),
    "multi-Si":      ("Photovoltaic panel, multi-Si, at plant",             "RER"),
    "perovskite-Si": ("Photovoltaic panel, perovskite-Si-tandem, at plant", "DE"),
    "ribbon-Si":     ("Photovoltaic panel, ribbon-Si, at plant",            "RER"),
    "single-Si":     ("Photovoltaic panel, single-Si, at plant",            "RER"),
}

# Set B - manufacturing geography, technology held constant
GEOGRAPHY = {
    f"{tech} | {loc}": (f"Photovoltaic panel, {tech}, at plant", loc)
    for tech in ("single-Si", "multi-Si")
    for loc in ("RER", "CN", "APAC", "US")
}

tech_acts = {label: one(*spec) for label, spec in TECHNOLOGY.items()}
geo_acts = {label: one(*spec) for label, spec in GEOGRAPHY.items()}

pd.DataFrame(
    [{"set": "A technology", "label": k, "activity": v["name"], "location": v["location"], "unit": v["unit"]}
     for k, v in tech_acts.items()]
    + [{"set": "B geography", "label": k, "activity": v["name"], "location": v["location"], "unit": v["unit"]}
       for k, v in geo_acts.items()]
)

In [ ]:
# the impact categories we report on
METHODS = [
    ("EF v3.1", "climate change", "global warming potential (GWP100)"),
    ("EF v3.1", "acidification", "accumulated exceedance (AE)"),
    ("EF v3.1", "energy resources: non-renewable", "abiotic depletion potential (ADP): fossil fuels"),
    ("EF v3.1", "material resources: metals/minerals",
     "abiotic depletion potential (ADP): elements (ultimate reserves)"),
    ("EF v3.1", "human toxicity: carcinogenic", "comparative toxic unit for human (CTUh)"),
    ("EF v3.1", "particulate matter formation", "impact on human health"),
    ("EF v3.1", "water use", "user deprivation potential (deprivation-weighted water consumption)"),
    ("EF v3.1", "land use", "soil quality index"),
]

GWP = METHODS[0]

# short labels + units, for axis titles later
METHOD_LABEL = {
    METHODS[0]: "climate change",
    METHODS[1]: "acidification",
    METHODS[2]: "fossil resources",
    METHODS[3]: "metals/minerals",
    METHODS[4]: "human tox. (cancer)",
    METHODS[5]: "particulate matter",
    METHODS[6]: "water use",
    METHODS[7]: "land use",
}
METHOD_UNIT = {mt: bd.Method(mt).metadata["unit"] for mt in METHODS}

pd.DataFrame(
    [{"short label": METHOD_LABEL[mt], "unit": METHOD_UNIT[mt], "in project": mt in bd.methods}
     for mt in METHODS]
)

---
## 7) One activity, one method
*(cheatsheet slide: "Calculating LCIA results for 1 activity & 1 method")*

This part of the cheatsheet is correct as printed and still works exactly as written.

In [ ]:
act = tech_acts["single-Si"]
ipcc = GWP

lca = bc.LCA({act: 1}, ipcc)   # {act: 1} is the functional unit
lca.lci()                      # builds the matrices, solves A s = f
lca.lcia()                     # characterisation: multiply the inventory by the CFs
lca.score                      # the LCIA score

In [ ]:
print(f"{act['name']} [{act['location']}]")
print(f"{lca.score:,.2f} {METHOD_UNIT[GWP]} per {act['unit']}")

### Beyond the cheatsheet: what `lca` now holds

Useful to know when you want numbers the `score` does not give you.

In [ ]:
pd.DataFrame([
    {"object": "demand_array",            "shape": lca.demand_array.shape},
    {"object": "technosphere_matrix (A)", "shape": lca.technosphere_matrix.shape},
    {"object": "biosphere_matrix (B)",    "shape": lca.biosphere_matrix.shape},
    {"object": "supply_array (s)",        "shape": lca.supply_array.shape},
    {"object": "inventory (g)",           "shape": lca.inventory.shape},
    {"object": "characterized_inventory", "shape": lca.characterized_inventory.shape},
])

In [ ]:
# switching method without re-solving the (expensive) technosphere system
for mt in METHODS[:4]:
    lca.switch_method(mt)
    lca.lcia()
    print(f"{METHOD_LABEL[mt]:<20} {lca.score:>14,.4g}  {METHOD_UNIT[mt]}")

---
## 8) Many activities × many methods
*(cheatsheet slide: "Calculating LCIA results for multiple activities (functional units)/methods")*

> ❌ **This is the biggest breakage on the cheatsheet.** The sequence
>
> ```python
> FU = [{x: 1} for x in acts]
> bd.calculation_setups["setupname"] = {"inv": FU, "ia": methods}
> mLCA = bc.MultiLCA("setupname")
> mLCA.results
> ```
>
> is the **Brightway 2 API**. In `bw2calc 2.x` `MultiLCA` was rewritten: it no longer reads a
> named calculation setup, it takes `demands`, `method_config` and `data_objs` directly, and the
> results live in `.scores` (a dict), not in `.results` (an array).
>
> `bd.calculation_setups` still exists, so the assignment line silently succeeds — which makes
> this failure especially confusing. Only the `bc.MultiLCA("setupname")` call raises.

In [ ]:
acts = list(tech_acts.values())
FU = [{x: 1} for x in acts]

# the assignment still works -- this is what makes the breakage confusing
bd.calculation_setups["pv_technologies"] = {"inv": FU, "ia": METHODS}
print("calculation_setups assignment: OK (but no longer used by bw2calc 2.x)")

try:
    mLCA = bc.MultiLCA("pv_technologies")            # as printed on the cheatsheet
except TypeError as err:
    print("bc.MultiLCA('pv_technologies') FAILS ->", type(err).__name__, err)

### The working Brightway 2.5 pattern

Three steps:

1. **`demands`** — a dict of `{label: {node_id: amount}}`. The label is yours to choose and
   becomes the key in the results, so make it readable.
2. **`method_config`** — `{"impact_categories": [...]}`.
3. **`data_objs`** — the datapackages, which `bd.get_multilca_data_objs()` assembles for you.

The payoff over looping `bc.LCA` is real: the technosphere matrix is factorised **once** and
reused for every functional unit.

In [ ]:
def multi_lca(activities, methods):
    """Run all activities x all methods, return a tidy DataFrame (rows = FU, cols = method)."""
    demands = {label: {act.id: 1} for label, act in activities.items()}
    config = {"impact_categories": list(methods)}

    data_objs = bd.get_multilca_data_objs(functional_units=demands, method_config=config)

    mlca = bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
    mlca.lci()
    mlca.lcia()

    # mlca.scores is {(method_tuple, fu_label): score}
    df = pd.Series(mlca.scores).unstack(level=0)
    df.index.name = "functional unit"
    return df.loc[list(activities), list(methods)]


tech_results = multi_lca(tech_acts, METHODS)
tech_results

In [ ]:
# more human-friendly labels, as the cheatsheet recommends
tech_table = tech_results.rename(columns={mt: f"{METHOD_LABEL[mt]}\n[{METHOD_UNIT[mt]}]" for mt in METHODS})
tech_table.style.format("{:,.4g}").background_gradient(cmap=BLUES, axis=0)

In [ ]:
# export to excel, e.g. for building figures elsewhere
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
tech_table.to_excel(out_dir / "pv_technology_results.xlsx")
print("written:", (out_dir / "pv_technology_results.xlsx").resolve())

---
## 8b) Reusing one LCA object — the "loop LCA"

`MultiLCA` is the right tool when you know all the functional units up front. But often you don't:
you want to loop, decide what to calculate next as you go, or keep the full `inventory` for each
run. Building a fresh `bc.LCA` every time is the obvious way — and the slow one, because **most of
the cost is factorising the technosphere matrix**, not the calculation itself.

The fix is to build **one** LCA object, factorise once, and then swap the demand and the method:

| call | what it re-does | what it keeps |
|---|---|---|
| `lca.lci(factorize=True)` | everything, once | — |
| `lca.redo_lci({new_id: 1})` | solves for a new demand | the **factorised** technosphere |
| `lca.switch_method(new_method)` | swaps the characterisation matrix | inventory, supply array |
| `lca.lcia()` | characterisation only | everything else |

Two traps:

1. `bc.LCA({act: 1}, ...)` accepts an **Activity object** at construction, but
   `redo_lci()` needs the **integer id**: `lca.redo_lci({act.id: 1})`. Passing the object raises
   a `KeyError` telling you to use `.id`.
2. `switch_method()` does *not* recalculate — you must call `lcia()` after it.

In [ ]:
# the "loop LCA": build once, factorize once, then reuse
first = tech_acts["single-Si"]

lca_loop = bc.LCA({first: 1}, GWP)
lca_loop.lci(factorize=True)      # <- factorize ONCE; this is where the time goes
lca_loop.lcia()

loop_rows = {}
for label, act in tech_acts.items():
    lca_loop.redo_lci({act.id: 1})          # NOTE: .id, not the Activity object
    for mt in METHODS:
        lca_loop.switch_method(mt)
        lca_loop.lcia()                      # switch_method does not recalculate
        loop_rows[(label, mt)] = lca_loop.score

loop_results = pd.Series(loop_rows).unstack()
loop_results.index.name = "functional unit"
loop_results.loc[list(tech_acts), list(METHODS)].round(4)

In [ ]:
# does it agree with MultiLCA? (it must, to machine precision)
diff = (loop_results.loc[list(tech_acts), list(METHODS)] - tech_results).abs()
rel = (diff / tech_results.abs()).to_numpy().max()
print(f"largest relative difference vs MultiLCA: {rel:.2e}")
assert rel < 1e-12, "loop LCA disagrees with MultiLCA"
print("identical results.")

### How much faster, really?

Four ways to get the same 7 activities × 4 methods, timed. Run it yourself — the ranking is
stable but the absolute numbers depend on your machine and database size.

In [ ]:
import time

bench_methods = METHODS[:4]
bench = {}

# a) the obvious way: a fresh LCA for every combination
t0 = time.time()
ref = {}
for label, act in tech_acts.items():
    for mt in bench_methods:
        lca_tmp = bc.LCA({act: 1}, mt)
        lca_tmp.lci()
        lca_tmp.lcia()
        ref[(label, mt)] = lca_tmp.score
bench["a) fresh bc.LCA every time"] = time.time() - t0

# b) the loop LCA
t0 = time.time()
lca_b = bc.LCA({first: 1}, bench_methods[0])
lca_b.lci(factorize=True)
lca_b.lcia()
got_b = {}
for label, act in tech_acts.items():
    lca_b.redo_lci({act.id: 1})
    for mt in bench_methods:
        lca_b.switch_method(mt)
        lca_b.lcia()
        got_b[(label, mt)] = lca_b.score
bench["b) one LCA + factorize + redo_lci"] = time.time() - t0

# c) MultiLCA
t0 = time.time()
demands = {label: {act.id: 1} for label, act in tech_acts.items()}
cfg = {"impact_categories": bench_methods}
dobjs = bd.get_multilca_data_objs(functional_units=demands, method_config=cfg)
mlca_c = bc.MultiLCA(demands=demands, method_config=cfg, data_objs=dobjs)
mlca_c.lci()
mlca_c.lcia()
got_c = {(fu, mt): v for (mt, fu), v in mlca_c.scores.items()}
bench["c) bc.MultiLCA"] = time.time() - t0

# d) FastScoresOnlyMultiLCA - no lci()/lcia(); .calculate() returns an xarray
t0 = time.time()
dobjs2 = bd.get_multilca_data_objs(functional_units=demands, method_config=cfg)
fast = bc.FastScoresOnlyMultiLCA(demands=demands, method_config=cfg, data_objs=dobjs2)
fast_arr = fast.calculate()
bench["d) bc.FastScoresOnlyMultiLCA"] = time.time() - t0

slowest = bench["a) fresh bc.LCA every time"]
pd.DataFrame(
    [{"approach": k, "seconds": round(v, 2), "speed-up": f"{slowest / v:.1f}x"}
     for k, v in bench.items()]
).set_index("approach")

In [ ]:
# all four agree
for name, got in [("loop LCA", got_b), ("MultiLCA", got_c)]:
    worst = max(abs(got[k] - ref[k]) / abs(ref[k]) for k in ref)
    print(f"{name:<12} max relative difference vs fresh-LCA: {worst:.2e}")

print("\nFastScoresOnlyMultiLCA returns an xarray, not a dict:")
print("  type:", type(fast_arr).__name__, "| dims:", fast_arr.dims, "| shape:", fast_arr.shape)

**Which to use.**

- **`MultiLCA`** — the default when you know the functional units in advance. Fastest here, and it
  keeps `supply_arrays`, `inventories` and `characterized_inventories` per functional unit.
- **The loop LCA** (`factorize=True` + `redo_lci`) — when the functional units are decided *inside*
  the loop, or when you want one full `lca` object to introspect between runs. Nearly as fast, and
  it is the pattern that underlies Monte Carlo and scenario loops.
- **`bc.FastScoresOnlyMultiLCA`** — built for very large or dense systems, using chunking and a
  pre-multiplied characterisation matrix. On a database this size it is *slower* than `MultiLCA`,
  so treat it as something to benchmark rather than assume. Note it has **no `lci()`/`lcia()`** —
  calling them raises `NotImplementedError`; use `.calculate()`, and the result is an
  `xarray.DataArray` indexed by method and functional unit rather than a dict.
- **`bc.CachingLCA`** — *do not use on `bw2calc 2.5.0`*: its supply-vector cache raises
  `TypeError: object of type 'int' has no len()` for any single-activity functional unit, which is
  the normal case. See section 15.

---
## 9) Plotting results
*(cheatsheet slide: "Plotting results – using pandas dataframes")*

The cheatsheet's `df.plot.bar(...)` works, but plotting eight impact categories on one axis is
misleading: `water use` is in the thousands while `human toxicity` is around 1e-8, so every bar
except one collapses to zero. Two fixes, in order of preference:

1. **One chart per question.** The headline indicator gets its own chart, at its own scale.
2. **Normalise, then use a heatmap.** For the all-categories overview, magnitude relative to the
   worst performer is the honest encoding — and a single-hue ramp reads it better than eight bars.

Never a second y-axis: two scales on one plot invent a relationship that is not in the data.

In [ ]:
# Why the naive version fails - the cheatsheet's own line, unnormalised
ax = tech_results.rename(columns=METHOD_LABEL).plot.bar(figsize=(12, 4), width=0.8)
ax.set_xlabel("")
ax.set_ylabel("impact score (raw)")
ax.set_title("Eight impact categories on one axis — unreadable by construction")
plt.xticks(rotation=0)
tidy(ax)
plt.show()

### 9.1 The headline: climate change per m² of panel

One measure, seven nominal categories → **one colour for every bar**, sorted, directly labelled.
Colouring the bars by their own value would double-encode the bar length and burn the only free
channel.

In [ ]:
gwp = tech_results[GWP].sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(gwp.index, gwp.values, color=SERIES[0], height=0.62)

for bar, value in zip(bars, gwp.values):
    ax.text(value + gwp.max() * 0.015, bar.get_y() + bar.get_height() / 2,
            f"{value:,.0f}", va="center", ha="left", fontsize=9, color=INK_2)

ax.set_xlabel(f"climate change  [{METHOD_UNIT[GWP]} per m² panel]")
ax.set_ylabel("")
ax.set_title("Carbon footprint of 1 m² PV panel, at plant")
ax.set_xlim(0, gwp.max() * 1.12)
tidy(ax, xgrid=True, ygrid=False)
plt.tight_layout()
plt.show()

### 9.2 The overview: all categories, normalised

Each column is divided by its own maximum, so `1.00` marks the worst performer in that category
and the cells become comparable. The absolute values stay visible as cell annotations, because a
normalised chart without them cannot be read back to real units.

In [ ]:
norm = tech_results / tech_results.max()          # column-wise, relative to the worst performer
norm_plot = norm.rename(columns=METHOD_LABEL)
annot = tech_results.rename(columns=METHOD_LABEL).map(lambda v: f"{v:,.3g}")

fig, ax = plt.subplots(figsize=(11, 4.4))
sns.heatmap(
    norm_plot, ax=ax, cmap=BLUES, vmin=0, vmax=1,
    annot=annot, fmt="", annot_kws={"fontsize": 8},
    linewidths=2, linecolor=SURFACE,
    cbar_kws={"label": "share of worst performer", "shrink": 0.75, "pad": 0.02},
)
ax.set_title("1 m² PV panel — all impact categories, normalised per category\n"
             "(cells annotated with absolute values)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Reading it.** a-Si is lowest in six of the eight categories, including climate change and land
use — but not in the other two, and the exceptions are large: its water use is more than an order
of magnitude above CIS, and micro-Si beats it on carcinogenic toxicity. So even a near-sweep is
not a clean win, which is exactly why you report a panel of indicators rather than carbon alone.

The perovskite-Si tandem is the worst performer in **all eight** categories (every cell in its row
is 1.00). Read that as a statement about the *inventory*, not about the technology's potential:
it represents a pilot-scale process, not a mature production line, so it carries pilot-scale
material and energy intensities throughout.

### 9.3 Where a normalised bar chart *is* the right form

When you want to see the shape of the trade-off per technology rather than the per-category
ranking, the cheatsheet's normalised bar chart works well — as small multiples, one panel per
technology, so no category is hidden behind another.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.6), sharey=True)

for ax, (label, row) in zip(axes.ravel(), norm_plot.iterrows()):
    ax.bar(range(len(row)), row.values, color=SERIES[0], width=0.68)
    ax.set_title(label, fontsize=10)
    ax.set_xticks(range(len(row)))
    ax.set_xticklabels(row.index, rotation=90, fontsize=7.5)
    ax.set_ylim(0, 1.05)
    tidy(ax)

axes.ravel()[-1].axis("off")
axes[0, 0].set_ylabel("share of worst performer")
axes[1, 0].set_ylabel("share of worst performer")
fig.suptitle("Impact profile per technology (each category normalised to its worst performer)",
             x=0.005, ha="left", fontsize=12, fontweight="semibold")
plt.tight_layout()
plt.show()

---
## 10) Geography: does it matter where the panel is made?

Set B holds the technology constant and varies only the manufacturing region. Two series
(single-Si, multi-Si) → two categorical slots, a legend, and one shared axis.

In [ ]:
geo_results = multi_lca(geo_acts, METHODS)

geo_gwp = geo_results[GWP].rename("gwp").reset_index()
geo_gwp[["technology", "region"]] = geo_gwp["functional unit"].str.split(" | ", regex=False, expand=True)
geo_wide = geo_gwp.pivot(index="region", columns="technology", values="gwp").loc[["RER", "US", "CN", "APAC"]]
geo_wide.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))

x = np.arange(len(geo_wide))
width = 0.36
for i, tech in enumerate(["single-Si", "multi-Si"]):
    offset = (i - 0.5) * (width + 0.02)          # 2px-equivalent gap between adjacent fills
    bars = ax.bar(x + offset, geo_wide[tech], width, label=tech, color=SERIES[i])
    ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=8.5, color=INK_2)

ax.set_xticks(x)
ax.set_xticklabels(geo_wide.index)
ax.set_xlabel("manufacturing region")
ax.set_ylabel(f"climate change  [{METHOD_UNIT[GWP]} / m²]")
ax.set_title("Same panel, different factory: carbon footprint of 1 m² PV panel by region")
ax.set_ylim(0, geo_wide.to_numpy().max() * 1.15)
ax.legend(title="", loc="upper left")
tidy(ax)
plt.tight_layout()
plt.show()

In [ ]:
# the spread, as a single number per technology
spread = pd.DataFrame({
    "min": geo_wide.min().round(1),
    "max": geo_wide.max().round(1),
    "best region": geo_wide.idxmin(),
    "worst region": geo_wide.idxmax(),
})
spread["worst vs best"] = (geo_wide.max() / geo_wide.min() - 1).map("{:+.1%}".format)

# for reference: the spread across technologies, all made in their own BAFU region
tech_gwp = tech_results[GWP]
print(f"technology spread (all 7) : {tech_gwp.min():.0f} - {tech_gwp.max():.0f} "
      f"kg CO2-eq/m2  (factor {tech_gwp.max() / tech_gwp.min():.2f})")
print(f"multi-Si vs single-Si, both RER: "
      f"{geo_wide.loc['RER', 'single-Si'] / geo_wide.loc['RER', 'multi-Si'] - 1:+.1%}")
spread

**Reading it.** Moving production of an *identical* panel from Europe to China raises its carbon
footprint by about 13 % — for both technologies, and with the four regions always in the same
order (RER < US < APAC < CN). That regional effect is **larger than the 9 % gap between multi-Si
and single-Si built in the same region**, so a crystalline-silicon technology ranking that leaves
the manufacturing location implicit is not a robust ranking. Across all seven technologies the
spread is wider — roughly a factor of two — so technology still matters more than geography
overall; it is *within* the crystalline-Si family that geography takes over. Section 11 isolates
the mechanism.

---
## 11) Contribution analysis
*(cheatsheet slide: "Contribution analysis")*

Three complementary tools — all working as advertised, with one fix to the cheatsheet's dataframe code.

### 11.1 `bw2analyzer` — the quickest look

Not on the cheatsheet, but this is the fastest way to see where a score comes from.
`print_recursive_calculation` walks the supply chain and prints anything above the cutoff.

In [ ]:
import bw2analyzer as ba

ba.print_recursive_calculation(tech_acts["single-Si"], GWP, max_level=3, cutoff=0.05)

In [ ]:
# top contributing processes (attributed by direct emissions) and top emitted flows
lca = bc.LCA({tech_acts["single-Si"]: 1}, GWP)
lca.lci()
lca.lcia()

ca = ba.ContributionAnalysis()

top_processes = pd.DataFrame(
    [{"score": s, "share": s / lca.score, "activity": a["name"], "location": a.get("location")}
     for s, _, a in ca.annotated_top_processes(lca, limit=12)]
)
top_processes.style.format({"score": "{:,.2f}", "share": "{:.1%}"})

In [ ]:
top_flows = pd.DataFrame(
    [{"score": s, "share": s / lca.score, "flow": f["name"],
      "compartment": "::".join(f.get("categories") or ())}
     for s, _, f in ca.annotated_top_emissions(lca, limit=10)]
)
top_flows.style.format({"score": "{:,.2f}", "share": "{:.1%}"})

**The mechanism, in one line:** `Hard coal, burned in power plant [CN]` alone carries about
28 % of a European single-Si panel's carbon footprint — because the *cells* in BAFU's European
panel come predominantly from Asia-Pacific. The panel is assembled in Europe; the emissions are
largely Chinese. This is what section 10's regional spread is made of.

### 11.2 `polyviz` — the cheatsheet's supply-chain table

The cheatsheet code works. One fix: `df.columns = [...]` assigns **ten** names, so the
concatenated frame must have ten columns — it does, but only if you keep the `+ d` prefix of
three items. The snippet below also adds the `if` guard that stops the loop crashing when a
combination returns nothing.

In [ ]:
from polyviz.utils import calculate_supply_chain

rows = []
for label, act in tech_acts.items():
    for mt in [GWP]:
        data = calculate_supply_chain(act, mt, 2, 1e-2)[0]   # level=2, cutoff=1 %
        if not data:
            continue
        rows.extend([[act["name"], act["location"], " - ".join(mt)] + d for d in data])

contrib = pd.DataFrame(rows, columns=[
    "activity", "location_activity", "lcia_method", "level", "contribution share",
    "absolute score", "amount_in_dataset", "activity_contributing",
    "location_contributing", "unit",
])
contrib.to_excel(out_dir / "pv_contribution.xlsx", index=False)
print(f"{len(contrib)} rows written to {out_dir / 'pv_contribution.xlsx'}")
contrib.head(10)

### 11.3 Contribution as a stacked bar

Level-1 contributors, grouped into the handful that matter plus an explicit `other`.
Stacked segments get a surface-coloured gap so adjacent fills never touch.

In [ ]:
KEEP = ["Photovoltaic cell", "Aluminium alloy", "Solar glass", "Electricity",
        "Photovoltaic laminate", "Transport"]

def group(name):
    for k in KEEP:
        if name.startswith(k):
            return k
    return "other"

lvl1 = contrib[contrib["level"] == 1].copy()
lvl1["group"] = lvl1["activity_contributing"].map(group)

stack = (lvl1.pivot_table(index="activity", columns="group",
                          values="absolute score", aggfunc="sum")
             .fillna(0))
# relabel rows from full activity name back to the short technology label
name_to_label = {a["name"]: k for k, a in tech_acts.items()}
stack.index = [name_to_label.get(i, i) for i in stack.index]

order = [c for c in KEEP if c in stack.columns] + (["other"] if "other" in stack.columns else [])
stack = stack[order].loc[tech_results[GWP].sort_values().index]
stack.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))

bottom = np.zeros(len(stack))
for i, col in enumerate(stack.columns):
    colour = "#b9b8b2" if col == "other" else SERIES[i]
    ax.bar(stack.index, stack[col], bottom=bottom, label=col,
           color=colour, width=0.62, edgecolor=SURFACE, linewidth=2)
    bottom += stack[col].to_numpy()

ax.set_ylabel(f"climate change  [{METHOD_UNIT[GWP]} / m²]")
ax.set_xlabel("")
ax.set_title("Which inputs carry the carbon footprint? (level-1 contributors, 1 % cutoff)")
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, 1.02))
ax.set_ylim(0, bottom.max() * 1.28)
tidy(ax)
plt.tight_layout()
plt.show()

---
## 12) Scenario analysis — manipulating datasets
*(cheatsheet slide: "Manipulating datasets/databases (w/o wurst)")*

> ❌ **Cheatsheet error.** The slide's snippet is Brightway 2 (`import brightway2 as bw`,
> `bw.Database(...)`) and — more importantly — it edits the **background database in place**.
> Never do that: the edit is irreversible without a re-import and silently changes every result
> you have already calculated.
>
> The pattern below does the same thing safely: copy the activity into a **separate foreground
> database**, edit the copy, and leave `bafu` untouched.

We test the mechanism section 11 identified — that the footprint is driven by where the *cells*
are made, not where the panel is assembled.

In [ ]:
if "pv-scenarios" in bd.databases:
    del bd.databases["pv-scenarios"]

fg = bd.Database("pv-scenarios")
fg.register()

base = tech_acts["single-Si"]                                        # RER panel, BAFU cell mix
cell_rer = one("Photovoltaic cell, single-Si, at plant", "RER")
cell_apac = one("Photovoltaic cell, single-Si, at plant", "APAC")
elec_ch = one("Electricity, medium voltage, production CH, at grid", "CH")

print("baseline panel:", base)
print("cell options  :", cell_rer, "|", cell_apac)

In [ ]:
def scenario(code, name, cell=None, electricity=None):
    """Copy the baseline panel into the foreground db and swap chosen inputs."""
    new = base.copy(code=code, database="pv-scenarios", name=name)
    for e in new.technosphere():
        if cell is not None and "Photovoltaic cell, single-Si" in e.input["name"]:
            e.input = cell
            e.save()
        if electricity is not None and "Electricity, medium voltage" in e.input["name"]:
            e.input = electricity
            e.save()
    return new


scenarios = {
    "S0 baseline (BAFU cell mix)":   base,
    "S1 cells from Europe":          scenario("s1", "PV panel single-Si, EU cells", cell=cell_rer),
    "S2 cells from Asia-Pacific":    scenario("s2", "PV panel single-Si, APAC cells", cell=cell_apac),
    "S3 EU cells + CH electricity":  scenario("s3", "PV panel single-Si, EU cells + CH elec",
                                              cell=cell_rer, electricity=elec_ch),
}

scen_results = multi_lca(scenarios, [GWP])[GWP]
scen_table = scen_results.to_frame("kg CO2-eq / m²")
scen_table["vs baseline"] = scen_results / scen_results.iloc[0] - 1
scen_table.style.format({"kg CO2-eq / m²": "{:,.1f}", "vs baseline": "{:+.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))

baseline = scen_results.iloc[0]
bars = ax.barh(scen_results.index[::-1], scen_results.values[::-1], color=SERIES[0], height=0.58)
ax.axvline(baseline, color=INK_2, linewidth=1, linestyle=(0, (4, 3)), zorder=3)
ax.text(baseline, len(scen_results) - 0.4, "  baseline", ha="left", va="center",
        fontsize=8.5, color=INK_2)

for bar, value in zip(bars, scen_results.values[::-1]):
    delta = value / baseline - 1
    ax.text(value + 2, bar.get_y() + bar.get_height() / 2,
            f"{value:,.0f}   ({delta:+.0%})" if delta else f"{value:,.0f}",
            va="center", ha="left", fontsize=9, color=INK_2)

ax.set_xlabel(f"climate change  [{METHOD_UNIT[GWP]} / m²]")
ax.set_title("Where the cells come from decides the footprint")
ax.set_xlim(0, scen_results.max() * 1.30)
tidy(ax, xgrid=True, ygrid=False)
plt.tight_layout()
plt.show()

**Reading it.** Sourcing cells in Europe cuts the footprint by about 16 %; additionally switching
the panel factory to Swiss electricity adds only a few more points, because the factory's own
electricity is a small share of the total. The lesson is generalisable: **edit where the impact
is, not where the activity is convenient to edit.** Contribution analysis tells you which one
that is.

In [ ]:
# tidy up: drop the foreground database and the demo calculation setup
del bd.databases["pv-scenarios"]
del bd.calculation_setups["pv_technologies"]

print("databases          :", list(bd.databases))
print("calculation setups :", list(bd.calculation_setups))

---
## 13) Uncertainty — Monte Carlo
*(cheatsheet slide: "Uncertainty analysis" — marked "To be filled")*

BAFU ships lognormal uncertainty on most exchanges, so this section fills that gap.

The important detail is **paired sampling**: running all functional units inside *one* `MultiLCA`
with `use_distributions=True` means every iteration draws one set of background parameters and
applies it to all panels. The differences between panels are then directly comparable, iteration
by iteration. Running separate Monte Carlos per panel and comparing the means throws that away.

In [ ]:
MC_ITERATIONS = 100          # ~1.5 s per iteration for 3 FUs; raise for real work
MC_SELECTION = ["a-Si", "multi-Si", "single-Si"]

mc_acts = {k: tech_acts[k] for k in MC_SELECTION}
demands = {label: {act.id: 1} for label, act in mc_acts.items()}
config = {"impact_categories": [GWP]}

data_objs = bd.get_multilca_data_objs(functional_units=demands, method_config=config)
mc = bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs,
                 use_distributions=True, seed_override=42)
mc.lci()
mc.lcia()

samples = []
for i in range(MC_ITERATIONS):
    if i:
        next(mc)
        mc.lcia()
    samples.append({label: mc.scores[(GWP, label)] for label in mc_acts})

mc_df = pd.DataFrame(samples)
mc_df.describe().round(1)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})

# left: the distributions
ax = axes[0]
parts = ax.violinplot([mc_df[c] for c in MC_SELECTION], showextrema=False, widths=0.75)
for body in parts["bodies"]:
    body.set_facecolor(SERIES[0])
    body.set_alpha(0.22)
    body.set_edgecolor("none")
for i, col in enumerate(MC_SELECTION, start=1):
    q1, med, q3 = np.percentile(mc_df[col], [25, 50, 75])
    ax.vlines(i, q1, q3, color=SERIES[0], linewidth=5, zorder=3)
    ax.plot(i, med, "o", color=SURFACE, markeredgecolor=SERIES[0],
            markeredgewidth=1.6, markersize=7, zorder=4)
    ax.plot(i, tech_results[GWP].loc[col], "D", color=SERIES[1], markersize=6, zorder=5)

# clip the long lognormal tail so the bulk of the distribution stays legible
ax.set_ylim(0, np.percentile(mc_df.to_numpy(), 99) * 1.05)
ax.set_xticks(range(1, len(MC_SELECTION) + 1))
ax.set_xticklabels(MC_SELECTION)
ax.set_ylabel(f"climate change  [{METHOD_UNIT[GWP]} / m²]")
ax.set_title(f"Monte Carlo, {MC_ITERATIONS} paired iterations")
ax.legend(handles=[
    Line2D([], [], color=SERIES[0], linewidth=5, label="interquartile range"),
    Line2D([], [], marker="o", linestyle="none", color=SURFACE,
           markeredgecolor=SERIES[0], markeredgewidth=1.6, markersize=7, label="median"),
    Line2D([], [], marker="D", linestyle="none", color=SERIES[1],
           markersize=6, label="deterministic score"),
], loc="upper left")
tidy(ax)

# right: the paired difference that actually answers the question
ax = axes[1]
diff = mc_df["single-Si"] - mc_df["a-Si"]
share = (diff > 0).mean()
ax.hist(diff, bins=24, color=SERIES[0], edgecolor=SURFACE, linewidth=1.2)
ax.axvline(0, color=INK_2, linewidth=1)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlabel(f"single-Si minus a-Si  [{METHOD_UNIT[GWP]} / m²]")
ax.set_ylabel("iterations")
ax.set_title(f"Paired difference - single-Si worse in {share:.0%} of iterations")
tidy(ax)

plt.tight_layout()
plt.show()

**Reading it.** The distributions overlap heavily — if you ran three independent Monte Carlos and
compared the means, you would conclude the technologies are indistinguishable. The **paired**
difference on the right says otherwise: because both panels share the same sampled background in
each iteration, single-Si comes out worse in the large majority of iterations. Overlapping
marginal distributions and a consistent paired ranking are not contradictory, and the paired view
is the one that answers "which should I choose?".

In [ ]:
# the comparison summarised as numbers
pd.DataFrame({
    "deterministic": tech_results[GWP].loc[MC_SELECTION],
    "MC median": mc_df.median(),
    "MC 5th pct": mc_df.quantile(0.05),
    "MC 95th pct": mc_df.quantile(0.95),
    "GSD (P95/P5)": (mc_df.quantile(0.95) / mc_df.quantile(0.05)),
}).round(2)

---
## 14) Reproducibility and collaboration
*(cheatsheet slide: "Project management: Reproducibility & collaboration")*

The cheatsheet's calls are correct for `bw2io 0.9.x`. Both are shown guarded — a BAFU-sized
project takes a while to pack.

```python
# back up the whole project (databases, methods, parameters) to a .tar.gz
bi.backup_project_directory(project="aalborg-rlcia-2026")           # -> your home folder
bi.backup_project_directory(project="aalborg-rlcia-2026",
                            dir_backup=r"C:\yourpath")              # -> somewhere else

# restore it, on any machine
bi.restore_project_directory(fp=r"C:\yourpath\brightway2-project-...-backup.tar.gz",
                             project_name="aalborg-rlcia-2026",
                             overwrite_existing=False)
```

In [ ]:
for fn in (bi.backup_project_directory, bi.restore_project_directory):
    print(f"{fn.__name__}{inspect.signature(fn)}\n")

In [ ]:
# uncomment to actually run the backup (BAFU-sized projects take a few minutes)
# backup_path = bi.backup_project_directory(project="aalborg-rlcia-2026")
# print(backup_path)

---
## 15) Cheatsheet corrections — summary

Everything checked against **`bw2data 4.7`**, **`bw2calc 2.5.0`**, **`bw2io 0.9.17`**
(the pinned course environment).

| # | Cheatsheet slide | As printed | Status | Correct form |
|---|---|---|---|---|
| 1 | Multiple activities/methods | `bc.MultiLCA("setupname")` after `bd.calculation_setups[...]` | **Broken** — bw2-only API | `bd.get_multilca_data_objs(functional_units=…, method_config=…)`, then `bc.MultiLCA(demands=…, method_config=…, data_objs=…)` |
| 2 | Multiple activities/methods | `mLCA.results` (array) | **Broken** | `mlca.scores` — a dict keyed `(method, fu_label)` |
| 2b | Appendix (command-window slide) | `bw2calc.multi_lca.calculation_setups["PVs"] = {...}` | **Broken** — `bw2calc.multi_lca` has no `calculation_setups` at all (`AttributeError`) | drop it; use the `MultiLCA` pattern in §8 |
| 3 | Reproducibility | `bd.projects.copy_projects(...)` | **Broken** — no such method | `bd.projects.copy_project(new_name, switch=True)` |
| 4 | Reproducibility | "copy the project to rename it, then delete the old one" | **Outdated** | `bd.projects.rename_project(new_name)` renames in place |
| 5 | Import databases | `if len(list(imp.unlinked) == 0):` | **Broken** — bracket misplaced; raises `TypeError: object of type 'bool' has no len()` | `if len(list(imp.unlinked)) == 0:` |
| 6 | Content of the dbs | `bd.Method(ilcd).metadata` where `ilcd` is a list | **Broken** — unhashable type | `bd.Method(ilcd[0]).metadata` |
| 7 | Installing/upgrading | `conda create -n yourenv –c conda-forge …` | **Broken on paste** — en-dash instead of `-` | Retype hyphens as ASCII `-c` |
| 8 | Reproducibility | `conda list -n envname --export C:\path\env.yml` | **Broken** — no redirect, wrong extension | `conda list -n envname --export > C:\path\env.txt`, or `conda env export -n envname > …yml` |
| 9 | Manipulating datasets | `import brightway2 as bw`; `bw.Database("ei35")` | **Outdated + unsafe** | `import bw2data as bd`; copy into a foreground database, never edit the background in place |
| 10 | Content of the dbs | `e.get('location')` on an exchange | **Works, but misleading** | `e.input["location"]` — states that you mean the input node |
| 11 | Plotting | `df.plot.bar(...)` across all impact categories | **Runs, but unreadable** | One chart per indicator, or normalise first (§9) |
| 12 | Contribution analysis | `calculate_supply_chain(...)` + 10 column names | **Works as printed** | — |
| 13 | 1 activity & 1 method | `bc.LCA({act: 1}, m)` → `.lci()` → `.lcia()` → `.score` | **Works as printed** | — |
| 14 | Manage databases | `.copy()`, `.rename()`, `del bd.databases[...]` | **Works as printed** | — |
| 15 | Projects | `set_current`, `create_project`, `delete_project`, `dir`, `report()` | **Works as printed** | — |
| 16 | Import databases | `bi.bw2setup()`, `SingleOutputEcospold2Importer`, `import_ecoinvent_release` | **Works as printed** | — |
| 17 | Reproducibility | `bi.backup_project_directory` / `restore_project_directory` | **Works as printed** | — |
| 18 | Searching | `db.search(...)`, `bio.search(..., filter={...})`, list comprehensions | **Works as printed** | — |
| 19 | Wurst | `wurst.extract_brightway2_databases`, `ws.contains/either/exclude/equals` | **Works** (wurst 0.5.3 installed) | — |
| 20 | *(not on the cheatsheet)* | `lca.lci(factorize=True)` + `redo_lci` + `switch_method` | **Works** — ~19x faster than a fresh `bc.LCA` per run | see §8b |
| 21 | *(not on the cheatsheet)* | `bc.FastScoresOnlyMultiLCA` | **Works, but** no `lci()`/`lcia()` (raises `NotImplementedError`); use `.calculate()`, returns an `xarray` | see §8b |
| 22 | *(not on the cheatsheet)* | `bc.CachingLCA` | **Broken in bw2calc 2.5.0** — `TypeError: object of type 'int' has no len()` for any single-activity FU | avoid; use §8b's loop LCA |

### Still marked "To be filled" on the cheatsheet

| Topic | Covered here |
|---|---|
| Uncertainty analysis | §13 — paired Monte Carlo with `use_distributions=True` |
| Scenario analysis | §12 — foreground copies with swapped inputs |
| Regionalisation | Day 2 notebooks (`edges` 1.4.0 is in the environment) |
| Parametrisation | `bw2parameters` is installed; see `D1-10` |
| Manipulating LCIA methods | `bi.ExcelLCIAImporter` — see `D1-04` §5 |

## Recap

You can now, using only the cheatsheet plus this notebook:

- set up, inspect, copy, rename and delete projects and databases
- import an Excel inventory and check its linking before writing it
- find activities by full-text search and by explicit list comprehension
- run a single LCA, and switch methods without re-solving the technosphere
- run many functional units against many methods with the **current** `MultiLCA` API
- turn results into DataFrames, export them, and plot them so they can actually be read
- find out where a score comes from, three different ways
- build scenarios safely in a foreground database
- quantify uncertainty with paired Monte Carlo, and read a paired comparison correctly

**The case study's answer.** Across seven PV technologies the carbon footprint of 1 m² of panel
spans roughly a factor of two (62 to 121 kg CO2-eq), and no technology is best in every impact
category. Moving the *same* panel between manufacturing regions shifts its footprint by about
13 % — more than the 9 % that separates multi-Si from single-Si made in the same region — and
contribution analysis shows why: the cell, and the coal-heavy electricity behind it, dominates.
Any crystalline-silicon comparison that fixes the technology but leaves the manufacturing
geography implicit is answering a different question than it appears to.